In [15]:
# NEO4J_URI="neo4j+s://427c9a4f.databases.neo4j.io"
# NEO4J_USERNAME="neo4j"
# NEO4J_PASSWORD="rQJMGc6onbUh6iWZ27jOg5QeyqBcMq3JWxUJ3_Y"

NEO4J_URI="neo4j://localhost:7687"
NEO4J_USERNAME="neo4j"
NEO4J_PASSWORD="admin-neo4j"

In [16]:
import os
os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD

In [17]:
from langchain_neo4j import Neo4jGraph
graph=Neo4jGraph(url=NEO4J_URI,username=NEO4J_USERNAME,password=NEO4J_PASSWORD)
graph

In [18]:
moview_query="""
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') | 
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') | 
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') | 
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))


"""
graph.query(moview_query)

[]

In [19]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")

ollama_llm = ChatOllama(
    model="qwen3-coder",
    # reasoning=False,
    temperature=0,
)
llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile")
llm

qa_llm = ChatOllama(
    model="granite4",
    temperature=0,
)

In [20]:
from langchain_neo4j import GraphCypherQAChain

chain=GraphCypherQAChain.from_llm(
    graph=graph,
    llm=ollama_llm,
    exclude_types=["Genre"],
    verbose=True,
    allow_dangerous_requests=True,
)
chain

GraphCypherQAChain(verbose=True, graph=<langchain_neo4j.graphs.neo4j_graph.Neo4jGraph object at 0x00000225BCDD7610>, cypher_generation_chain=PromptTemplate(input_variables=['examples', 'question', 'schema'], input_types={}, partial_variables={}, template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nExamples (optional):\n{examples}\n\nThe question is:\n{question}')
| _ChatModelBinding(bound=ChatOllama(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, model='qwen3-coder', temperature=0.0), kwargs={}, con

In [21]:
chain.graph_schema

'Node properties:\nMovie {imdbRating: FLOAT, id: STRING, released: DATE, title: STRING}\nPerson {name: STRING}\nRelationship properties:\n\nThe relationships:\n(:Person)-[:DIRECTED]->(:Movie)\n(:Person)-[:ACTED_IN]->(:Movie)'

In [22]:
examples = [
    {
        "question": "How many artists are there?",
        "query": "MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)",
    },
    {
        "question": "Which actors played in the movie Casino?",
        "query": "MATCH (m:Movie {{title: 'Casino'}})<-[:ACTED_IN]-(a) RETURN a.name",
    },
    {
        "question": "How many movies has Tom Hanks acted in?",
        "query": "MATCH (a:Person {{name: 'Tom Hanks'}})-[:ACTED_IN]->(m:Movie) RETURN count(m)",
    },
    {
        "question": "Find actors who have acted in more than one movie.",
        "query": "MATCH (a:Person)-[:ACTED_IN]->(m:Movie) WITH a, COUNT(m) AS movieCount WHERE movieCount > 1 RETURN a.name ORDER BY movieCount DESC",
    },
    {
        "question": "List all the genres of the movie Schindler's List",
        "query": "MATCH (m:Movie {{title: 'Schindler\\'s List'}})-[:IN_GENRE]->(g:Genre) RETURN g.name",
    },
    {
        "question": "Which actors have worked in movies from both the comedy and action genres?",
        "query": "MATCH (a:Person)-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g1:Genre), (a)-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g2:Genre) WHERE g1.name = 'Comedy' AND g2.name = 'Action' RETURN DISTINCT a.name",
    },
    {
        "question": "Which directors have made movies with at least three different actors named 'John'?",
        "query": "MATCH (d:Person)-[:DIRECTED]->(m:Movie)<-[:ACTED_IN]-(a:Person) WHERE a.name STARTS WITH 'John' WITH d, COUNT(DISTINCT a) AS JohnsCount WHERE JohnsCount >= 3 RETURN d.name",
    },
    {
        "question": "Identify movies where directors also played a role in the film.",
        "query": "MATCH (p:Person)-[:DIRECTED]->(m:Movie), (p)-[:ACTED_IN]->(m) RETURN m.title, p.name",
    },
    {
        "question": "Find the actor with the highest number of movies in the database.",
        "query": "MATCH (a:Person)-[:ACTED_IN]->(m:Movie) RETURN a.name, COUNT(m) AS movieCount ORDER BY movieCount DESC LIMIT 1",
    },
]

In [23]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

example_prompt = PromptTemplate.from_template(
    "User input:{question}\n Cypher query:{query}"
)

prompt = FewShotPromptTemplate(
    examples=examples[:5],
    example_prompt=example_prompt,
    prefix=(
        "You are a Neo4j expert. Given an input question, generate a single syntactically correct Cypher query.\n"
        "Schema:\n{schema}\n"
        "Rules:\n"
        "- Output ONLY the Cypher query. No explanations, no alternatives, no markdown.\n"
        "- Never use HAVING. Filter aggregated results using WITH ... WHERE instead.\n"
        "- Never use size() with patterns. Use COUNT {{ (pattern) }} subquery instead."
    ),
    suffix="User input: {question}\nCypher query: ",
    input_variables=["question", "schema"],
)

In [24]:
prompt

FewShotPromptTemplate(input_variables=['question', 'schema'], input_types={}, partial_variables={}, examples=[{'question': 'How many artists are there?', 'query': 'MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)'}, {'question': 'Which actors played in the movie Casino?', 'query': "MATCH (m:Movie {{title: 'Casino'}})<-[:ACTED_IN]-(a) RETURN a.name"}, {'question': 'How many movies has Tom Hanks acted in?', 'query': "MATCH (a:Person {{name: 'Tom Hanks'}})-[:ACTED_IN]->(m:Movie) RETURN count(m)"}, {'question': 'Find actors who have acted in more than one movie.', 'query': 'MATCH (a:Person)-[:ACTED_IN]->(m:Movie) WITH a, COUNT(m) AS movieCount WHERE movieCount > 1 RETURN a.name ORDER BY movieCount DESC'}, {'question': "List all the genres of the movie Schindler's List", 'query': "MATCH (m:Movie {{title: 'Schindler\\'s List'}})-[:IN_GENRE]->(g:Genre) RETURN g.name"}], example_prompt=PromptTemplate(input_variables=['query', 'question'], input_types={}, partial_variables={}, te

In [25]:
print(prompt.format(question="How many artists are there?", schema="foo"))

You are a Neo4j expert. Given an input question, generate a single syntactically correct Cypher query.
Schema:
foo
Rules:
- Output ONLY the Cypher query. No explanations, no alternatives, no markdown.
- Never use HAVING. Filter aggregated results using WITH ... WHERE instead.
- Never use size() with patterns. Use COUNT { (pattern) } subquery instead.

User input:How many artists are there?
 Cypher query:MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)

User input:Which actors played in the movie Casino?
 Cypher query:MATCH (m:Movie {title: 'Casino'})<-[:ACTED_IN]-(a) RETURN a.name

User input:How many movies has Tom Hanks acted in?
 Cypher query:MATCH (a:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie) RETURN count(m)

User input:Find actors who have acted in more than one movie.
 Cypher query:MATCH (a:Person)-[:ACTED_IN]->(m:Movie) WITH a, COUNT(m) AS movieCount WHERE movieCount > 1 RETURN a.name ORDER BY movieCount DESC

User input:List all the genres of the movie Sc

In [26]:
chain=GraphCypherQAChain.from_llm(
    graph=graph,
    cypher_llm=ollama_llm,  # qwen3-coder: sinh Cypher query
    qa_llm=qa_llm,             # Groq: trả lời ngôn ngữ tự nhiên từ kết quả
    cypher_prompt=prompt,
    verbose=True,
    allow_dangerous_requests=True,
)

In [27]:
chain.invoke("Which actors played in the movie Casino?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie {title: 'Casino'})<-[:ACTED_IN]-(a) RETURN a.name
Full Context:
[{'a.name': 'James Woods'}, {'a.name': 'Joe Pesci'}, {'a.name': 'Robert De Niro'}, {'a.name': 'Sharon Stone'}]

> Finished chain.


{'query': 'Which actors played in the movie Casino?',
 'result': 'James Woods, Joe Pesci, Robert De Niro, and Sharon Stone played in the movie Casino.'}

In [28]:
chain.invoke("actors who acted in multiple movies")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (a:Person)-[:ACTED_IN]->(m:Movie) WITH a, COUNT(m) AS movieCount WHERE movieCount > 1 RETURN a.name ORDER BY movieCount DESC
Full Context:
[{'a.name': 'John Travolta'}, {'a.name': 'Gene Hackman'}, {'a.name': 'Julianne Moore'}, {'a.name': 'Robert Downey Jr.'}, {'a.name': 'Christian Slater'}, {'a.name': 'Samuel L. Jackson'}, {'a.name': 'Gary Oldman'}, {'a.name': 'Angela Bassett'}, {'a.name': 'Robert De Niro'}, {'a.name': 'Julia Ormond'}]

> Finished chain.


{'query': 'actors who acted in multiple movies',
 'result': 'John Travolta, Gene Hackman, Julianne Moore, Robert Downey Jr., Christian Slater, Samuel L. Jackson, Gary Oldman, Angela Bassett, Robert De Niro, and Julia Ormond have all acted in multiple movies.'}